# Reload

In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
import random
import datetime

from time_slot import TimeSlot
from locations import Locations
from truck import *
from freight import Freight

#Imports for the AI algorithms
import numpy as np
import gymnasium as gym
from gymnasium import spaces

import os
import time

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.ppo import MlpPolicy

#Imports for the imitation learning
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.util.util import make_vec_env

from stable_baselines3.common.evaluation import evaluate_policy

from imitation.data import rollout

from imitation.algorithms import bc

from imitation.algorithms import preference_comparisons
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from imitation.policies.base import FeedForward32Policy, NormalizeFeaturesExtractor
from imitation.rewards.reward_wrapper import RewardVecEnvWrapper

from imitation.data.types import Trajectory, Transitions
import pickle

from imitation.algorithms.adversarial.gail import GAIL
from imitation.algorithms.adversarial.airl import AIRL
from imitation.rewards.reward_nets import BasicShapedRewardNet

import tempfile
from imitation.algorithms.dagger import SimpleDAggerTrainer

#Explainability
import sklearn
import sklearn.datasets
import sklearn.ensemble
import lime
import lime.lime_tabular
from __future__ import print_function
import pandas as pd
from sklearn.utils import Bunch
import torch

#S2E
from S2E_rewards import Rewards

#GUI
import tkinter as tk
from tkinter import ttk

# Parameters

In [3]:
number_of_trucks = 125
number_of_freight = 250

# Time Frame

In [4]:
start_time = datetime.datetime.now() 
print("start_time =", start_time)

#Time frame of 1 week
number_of_days = 7

delta_time = datetime.timedelta(days=1) 
delta_time_seconds_day = 24 * 60 * 60 #Seconds in a day
delta_time_seconds_week = number_of_days * 24 * 60 * 60 #Seconds in a Week

time_frame = []

for i in range (number_of_days):
    end_time = start_time + delta_time
    
    time_frame += [start_time]
    start_time = end_time

#Return start_time back to normal
start_time -= datetime.timedelta(seconds=delta_time_seconds_week)
print("start_time =", start_time)
    
print("time_frame =", time_frame, end = '\n\n')

days_of_the_week = []
for i in range(number_of_days):
    days_of_the_week.append(i)

start_time = 2026-03-19 14:57:05.998496
start_time = 2026-03-19 14:57:05.998496
time_frame = [datetime.datetime(2026, 3, 19, 14, 57, 5, 998496), datetime.datetime(2026, 3, 20, 14, 57, 5, 998496), datetime.datetime(2026, 3, 21, 14, 57, 5, 998496), datetime.datetime(2026, 3, 22, 14, 57, 5, 998496), datetime.datetime(2026, 3, 23, 14, 57, 5, 998496), datetime.datetime(2026, 3, 24, 14, 57, 5, 998496), datetime.datetime(2026, 3, 25, 14, 57, 5, 998496)]



# Locations

In [5]:
locations = Locations.locations

# Trucks

In [6]:
def generateRandomTrucks(number_of_trucks):
    trucks = []
    
    #Full list of haulier companies at at https://www.imt-ip.pt/sites/IMTT/Portugues/TransportesRodoviarios/EmpresasLicenciadas/Documents/Empresas_Mercadorias_OrdEmp.pdf
    companies = ["CARKARMA", "CASTRO E MERINO TRANSPORTES", "DESTINOS VIBRANTES, UNIPESSOAL", "DROPLOG", "ESTORILMUDA - TRANSPORTES", "MAIN ROAD CONSTRUCTIONS", "SILVA & AZEVEDO", "TRANSPORTES JOAQUIM MONTEIRO", "TRANSPORTES MALHADA", "TRANSPORTES RAIO", "WINDY TRANSPORT-SUCURSAL EM PORTUGAL"]
    
    truck_types = ["20' Standard - Steel", "40' Standard - Steel", "40' High - Steel", "45' High - Steel", "20' Standard Reefer", "40' High Cube Standard Reefer", "40' High Cube Magnum Reefer", "40' High Cube StarCare", "40' High Cube Super Freezer", "20' Open Top", "40' Open Top High Cube", "40' High Flat Rack Collapsible"]

    for current_id in range(number_of_trucks):
        company = random.choice(companies)
        location = random.choice(locations)

        availability = {}
        
        number_of_time_slots = random.randint(3, 7) #We are considering the time frame of 1 week
        time_ids = random.sample(days_of_the_week, k = number_of_time_slots)

        for time_id in time_ids:
            #from https://stackoverflow.com/questions/553303/how-to-generate-a-random-date-between-two-other-dates?page=1&tab=scoredesc#tab-top
            random_second_start = random.randrange(delta_time_seconds_day) #Random time between the two dates
            new_time_slot_start = time_frame[time_id] + datetime.timedelta(seconds=random_second_start)

            random_second_end = random.randrange(delta_time_seconds_day) #Random time between the two dates
            new_time_slot_end = time_frame[time_id] + datetime.timedelta(seconds=random_second_end)

            if random_second_start < random_second_end: #Start has to come before the end
                new_time_slot = TimeSlot(time_id, new_time_slot_start, new_time_slot_end)
            else: #Start doesn't come before the end, so switch places
                new_time_slot = TimeSlot(time_id, new_time_slot_end, new_time_slot_start)

            availability[time_id] = new_time_slot

        #emission rate in gCO2/t-km; between 58.3 and 307.2 according to https://theicct.org/publication/co2-emissions-from-trucks-in-the-eu-an-analysis-of-the-heavy-duty-co2-standards-baseline-data/
        emission_rate = round(random.uniform(58.3, 307.3), 1)
        
        #Choose type of truck
        #truck_type = random.choice(truck_types)
        truck_type = random.choices(truck_types, cum_weights = [1, 2, 3, 4, 4.8, 5.6, 6.4, 7.2, 8, 9.33, 10.66, 11.99], k = 1)[0]
        
        if (truck_type == "20' Standard - Steel"):
            truck = Dry_20StandardSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' Standard - Steel"):
            truck = Dry_40StandardSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High - Steel"):
            truck = Dry_40HighSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "45' High - Steel"):
            truck = Dry_45HighSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "20' Standard Reefer"):
            truck = Refrigerated_20StandardReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Standard Reefer"):
            truck = Refrigerated_40HighCubeStandardReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Magnum Reefer"):
            truck = Refrigerated_40HighCubeMagnumReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube StarCare"):
            truck = Refrigerated_40HighCubeStarCare(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Super Freezer"):
            truck = Refrigerated_40HighCubeSuperFreezer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "20' Open Top"):
            truck = Special_20OpenTop(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' Open Top High Cube"):
            truck = Special_40OpenTopHighCube(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Flat Rack Collapsible"):
            truck = Special_40HighFlatRackCollapsible(current_id, company, location, availability, emission_rate)
        
        trucks.append(truck)
        
    return trucks
    


# Freight

In [7]:
def generateRandomFreight(number_of_freight):
    freight_list = []
    
    categories = ["Dry", "Refrigerated", "Special"]

    current_id = 0

    for i in range(number_of_freight):
        category = random.choice(categories)
        
        if (category == "Dry"):
            weight = random.uniform(24, 28.87) #28.87 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 13555) #13556 = length of the container with the most length
            width = random.uniform(2000, 2351) #2352 = width of the widest container
            height = random.uniform(1800, 2697) #2698 = height of the tallest container
        elif (category == "Refrigerated"):
            weight = random.uniform(24, 29.67) #29.67 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 11587) #11588 = length of the container with the most length
            width = random.uniform(2200, 2289) #2290 = width of the widest container
            height = random.uniform(1700, 2504) #2505 = height of the tallest container
        elif (category == "Special"):
            weight = random.uniform(15, 47.3) #47.3 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 12031) #12032 = length of the container with the most length
            width = random.uniform(2300, 2373) #2374 = width of the widest container
            height = random.uniform(1700, 2652) #2653 = height of the tallest container
        
        origin = random.choice(locations)
        destination = random.choice(locations)
        #Guarantee that origin is different than the destination
        while (origin == destination):
            destination = random.choice(locations)

        random_second_arrival = random.randrange(delta_time_seconds_week) #Random time between the two dates
        new_arrival = start_time + datetime.timedelta(seconds=random_second_arrival)
        
        random_second_deadline = random.randrange(delta_time_seconds_week) #Random time between the two dates
        new_deadline = start_time + datetime.timedelta(seconds=random_second_deadline)
        
        if random_second_deadline > random_second_arrival: #Arrival needs to happen first
            arrival = new_arrival
            deadline = new_deadline
        else:
            arrival = new_deadline
            deadline = new_arrival
        
        freight = Freight(current_id, category, weight, length, width, height, origin, destination, arrival, deadline)
        freight_list.append(freight)
        
        current_id += 1
    
    return freight_list

# Calculate Statistics

In [8]:
def calculateStatistics(trucks):
    total_distance_travelled = 0
    total_distance_travelled_with_no_freight = 0
    total_time_spent = datetime.timedelta(seconds=0)
    total_time_spent_with_no_freight = datetime.timedelta(seconds=0)
    total_grams_of_CO2_emitted = 0

    number_of_freight_delivered = 0

    for truck in trucks:
        timeslots = truck.getTimeSlots()

        for day in days_of_the_week:
            slot = timeslots.get(day)

            if ( (slot) and (not slot.isAvailable()) ):
                total_distance_travelled += slot.getDistanceTravelled()
                total_distance_travelled_with_no_freight += slot.getDistanceTravelledWithNoFreight()
                total_time_spent += slot.getTimeSpent()
                total_time_spent_with_no_freight += slot.getTimeSpentWithNoFreight()
                total_grams_of_CO2_emitted += slot.getGramsOfCO2Emitted()

                number_of_freight_delivered += 1
                
    print("total_distance_travelled:", round (total_distance_travelled, 1), "Km")
    print("total_distance_travelled_with_no_freight:", round(total_distance_travelled_with_no_freight, 1), "Km\n")

    print("total_time_spent:", total_time_spent)
    print("total_time_spent_with_no_freight:", total_time_spent_with_no_freight, end = "\n\n")

    total_grams_of_CO2_emitted = round(total_grams_of_CO2_emitted)
    print("total_grams_of_CO2_emitted:", f"{total_grams_of_CO2_emitted:_}", "grams\n")

    print("number_of_freight_delivered:", number_of_freight_delivered)
    print("percentage of freight delivered:", round( (number_of_freight_delivered / number_of_freight * 100), 1), "%")
    print("\n\n")

# AI Algorithm - Reinforcement Learning

## Create a Custom Environment

In [9]:
class FreightTruckMatchingEnv(gym.Env):
    """Custom Environment that follows gym interface."""

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(2) #Good time slot or bad time slot

        self.observation_space = spaces.Box(low=-1, high=np.inf,
                                            shape=(3,), dtype=np.float64)
        
        self.S2E_reward = Rewards()

        
    def on_left_click_exit(self, _):
        self.window.destroy()
        self.done = True
    
    def on_left_click_auto(self, _):
        self.auto = True
        self.window.destroy()
        
    def on_left_click_go_back(self, _):
        #Clear the window
        for widget in self.window.winfo_children():
            widget.destroy()
                    
        self.window.title('Select your Preferred Matching')  

        if (self.accepted_view):
            text = 'Truck-Freight Matchings Accepted by the AI'
            fg = 'green'
        else:
            text = 'Truck-Freight Matchings Rejected by the AI'
            fg = 'red'
            
        self.label = tk.Label(self.window, text=text, fg = fg, font=("Helvetica", 24))
        self.label.pack()

        observations = []
        self.current_trucks = []
        self.current_slots = []

        slots_truck_list = []
        if (self.accepted_view):
            slots_truck_list = self.good_slots_truck_list
        else:
            slots_truck_list = self.bad_slots_truck_list
            
        for slot in slots_truck_list:
            observations.append(slot[2])
            self.current_trucks.append(slot[1])
            self.current_slots.append(slot[0])

        #treeview
        self.table = ttk.Treeview(self.window, columns = ('distance', 'distance_no_freight', 'CO2'), show = 'headings')
        self.table.heading('distance', text = 'Distance Travelled (Km)')
        self.table.heading('distance_no_freight', text = 'Distance Travelled with no Freight (Km)')
        self.table.heading('CO2', text = 'Grams of CO2 Emitted')
        self.table.pack(fill = 'both', expand = True)

        #Insert values
        for obs in observations:
            obs = np.round(np.exp(obs) - 1, 1) #Retrieve the original value
            self.table.insert(parent = '', index = tk.END, values = (obs[0], obs[1], f"{obs[2]:_}"))

        #Table Events
        self.table.bind('<<TreeviewSelect>>', self.item_select)

        if (self.accepted_view):
            text = "Switch to Rejected Matchings"
            fg = 'red'
        else:
            text = "Switch to Accepted Matchings"
            fg = 'green'
        
        self.b1 = tk.Button(self.window, text=text, fg=fg, font=("Helvetica", 16))
        self.b1.bind("<Button-1>", self.on_left_click_switch)
        self.b1.pack()

        #self.b2 = tk.Button(self.window, text='Manual (click to switch to Auto)', font=("Helvetica", 16))
        #self.b2.bind("<Button-1>", self.on_left_click_auto)
        #self.b2.pack()
        
        self.b2 = tk.Button(self.window, text='Exit Program', font=("Helvetica", 16))
        self.b2.bind("<Button-1>", self.on_left_click_exit)
        self.b2.pack()

    def on_left_click_select_matching(self, _):
        if (self.accepted_view):
            self.slots_truck_list = self.good_slots_truck_list
        else:
            self.slots_truck_list = self.bad_slots_truck_list
        
        self.best_slot_truck = self.slots_truck_list[self.index]
        
        self.window.destroy()
    
    def item_select(self, _):
        for i in self.table.selection():
            self.values = self.table.item(i)['values']
            
        #Find index of the clicked item
        items = self.table.get_children()
        self.index = items.index(self.table.selection()[0])
        
        #Get the truck's company
        truck = self.current_trucks[self.index]
        truck_company = truck.getCompany()
        
        #Get the time spent travelling
        slot = self.current_slots[self.index]
        time_spent_travelling = slot["travel_time"]
        time_spent_travelling_no_freight = slot["time_spent_with_no_freight"]
        
        #Clear the window
        for widget in self.window.winfo_children():
            widget.destroy()
            
        self.window.title('Truck-Freight Match')  
        
        #treeview
        self.table = ttk.Treeview(self.window, columns = ('distance', 'distance_no_freight', 'CO2'), show = 'headings', height = 1)
        self.table.heading('distance', text = 'Distance Travelled (Km)')
        self.table.heading('distance_no_freight', text = 'Distance Travelled with no Freight (Km)')
        self.table.heading('CO2', text = 'Grams of CO2 Emitted')
        self.table.pack(fill = 'both', expand = True)

        #Insert values
        self.table.insert(parent = '', index = tk.END, values = (self.values[0], self.values[1], self.values[2]))
        
        #Second Table
        self.table2 = ttk.Treeview(self.window, columns = ('time', 'time_no_freight', 'company'), show = 'headings', height = 1)
        self.table2.heading('time', text = 'Travel Time (HH:mm:ss)')
        self.table2.heading('time_no_freight', text = 'Travel Time with no Freight (HH:mm:ss)')
        self.table2.heading('company', text = 'Truck Company')
        self.table2.pack(fill = 'both', expand = True)

        #Insert values
        self.table2.insert(parent = '', index = tk.END, values = (time_spent_travelling, time_spent_travelling_no_freight, truck_company))
        
        #Third Table
        truck_location = truck.getLocation()
        freight = self.freight_list[self.freight_ind] #Get the current freight
        pickup = freight.getOrigin()
        destination = freight.getDestination()
        
        self.table3 = ttk.Treeview(self.window, columns = ('truck', 'pickup', 'destination'), show = 'headings', height = 1)
        self.table3.heading('truck', text = 'Truck Location')
        self.table3.heading('pickup', text = 'Pickup Location')
        self.table3.heading('destination', text = 'Destination')
        self.table3.pack(fill = 'both', expand = True)

        #Insert values
        self.table3.insert(parent = '', index = tk.END, values = (truck_location, pickup, destination))
        
        #Label
        label_text = "Status: Matching "
        if (self.accepted_view):
            label_text += 'Accepted'
            fg = 'green'
        else:
            label_text += 'Rejected'
            fg = 'red'
        label = tk.Label(self.window, text=label_text, fg = fg, font=("Helvetica", 12))
        label.pack()
        
        #################################################
        
        #Why did the AI accept/reject?
        
        if (self.accepted_view):
            text = 'Why did the AI accept?'
        else:
            text = 'Why did the AI reject?'
        label = tk.Label(self.window, text=text, bg = "white", font=("Helvetica", 12))
        label.pack()
        
        #Get the log values
        distance = np.log(float(self.values[0]) + 1)
        distance_no_freight = np.log(float(self.values[1]) + 1)
        grams_CO2 = np.log(float(self.values[2]) + 1)
        
        state = [distance, distance_no_freight, grams_CO2]
        
        match_evaluation, explanations, self.predicted_concept = self.S2E_reward.getMatchEvaluation(state)
        
        text = f"This matching is {match_evaluation} because it leads to:"
        
        for explanation in explanations:
            text += "\n"
            text += explanation
            
        if (match_evaluation == "great"):
            bg = "green"
        elif (match_evaluation == "good"):
            bg = "green"
        elif (match_evaluation == "bad"):
            bg = "red"
        elif (match_evaluation == "terrible"):
            bg = "red"
        else:
            bg = "white"
            
        label = tk.Label(self.window, text=text, bg = bg, fg = 'white', font=("Helvetica", 12))
        label.pack()
        
        
        #################################################
        #How does the model get to its decision?
        
        label_text = "How does the model get to its decision?"
        label = tk.Label(self.window, text=label_text, bg="white", font=("Helvetica", 12))
        label.pack()
        
        label_text = "The model was trained on 150,000 scenarios, many of which are similar to this one. As such, it can predict the most likely outcome of a matching."
        label = tk.Label(self.window, text=label_text, font=("Helvetica", 12))
        label.pack()
        
        #################################################
        #Similar Training Examples
        label_text = "3 matches classified as similar:"
        label = tk.Label(self.window, text=label_text, font=("Helvetica", 12))
        label.pack()
        
        #Second Table
        self.table2 = ttk.Treeview(self.window, columns = ('distance', 'distance_no_freight', 'CO2'), show = 'headings', height = 1)
        self.table2.heading('distance', text = 'Distance Travelled (Km)')
        self.table2.heading('distance_no_freight', text = 'Distance Travelled with no Freight (Km)')
        self.table2.heading('CO2', text = 'Grams of CO2 Emitted')
        self.table2.pack(fill = 'both', expand = True)

        #Find 3 training examples with the same concept
        state_arr = np.load("datasets/data/state_aligned.npy", allow_pickle=True)
        explanation_arr = np.load("datasets/data/explanation_aligned.npy", allow_pickle=True)
        
        training_examples = []
        for i in range ( len(state_arr) ):
            explanation = explanation_arr[i]
            
            training_example_concept = self.S2E_reward.get_shorthand_concept(explanation)
            
            if (training_example_concept == self.predicted_concept):
                state = state_arr[i]
                training_examples.append( np.round(np.exp(state) - 1, 1) ) #Retrieve the original value
                
                if ( len(training_examples) >= 3 ):
                    break
        
        #Insert values
        for training_example in training_examples:
            self.table2.insert(parent = '', index = tk.END, values = (training_example[0], training_example[1], f"{training_example[2]:_}"))
        
        #################################################
        
        b1 = tk.Button(self.window, text='Select this matching', bg='orange', font=("Helvetica", 16))
        b1.bind("<Button-1>", self.on_left_click_select_matching)
        b1.pack()
            
        b3 = tk.Button(self.window, text='Go Back', font=("Helvetica", 16))
        b3.bind("<Button-1>", self.on_left_click_go_back)
        b3.pack()
            
            
    def on_left_click_switch(self, _):
        self.table.delete(*self.table.get_children())
        
        observations = []
        self.current_trucks = []
        self.current_slots = []
                    
        if (self.accepted_view):
            for slot in self.bad_slots_truck_list:
                observations.append(slot[2])
                self.current_trucks.append(slot[1])
                self.current_slots.append(slot[0])
            
            self.accepted_view = False
            self.b1.config(text = "Switch to Accepted Matchings", fg = 'green')
            self.label.config(text = "Truck-Freight Matchings Rejected by the AI", fg = 'red', font=("Helvetica", 24))
        else:
            for slot in self.good_slots_truck_list:
                observations.append(slot[2])
                self.current_trucks.append(slot[1])
                self.current_slots.append(slot[0])
                
            self.accepted_view = True
            self.b1.config(text = "Switch to Rejected Matchings", fg = 'red')
            self.label.config(text = "Truck-Freight Matchings Accepted by the AI", fg = 'green', font=("Helvetica", 24))

        #Insert values
        for obs in observations:
            obs = np.round(np.exp(obs) - 1, 1) #Retrieve the original value
            self.table.insert(parent = '', index = tk.END, values = (obs[0], obs[1], f"{obs[2]:_}"))
        
        
    def step(self, action):            
        ###########################Reward the Agent###########################
        
        self.reward = 0.0
        
        if (self.slot): #The slot was available
            self.reward, predicted_concept = self.S2E_reward.return_reward(self.observation, action)
            
            #distance_travelled = self.slot["distance_travelled"]
            #distance_travelled_with_no_freight = self.slot["distance_travelled_with_no_freight"]
            #grams_of_CO2_emitted = self.slot["grams_of_CO2_emitted"]
        
            if (action == 0): #actions: 0 = bad choice, 1 = good choice, according to the agent
                #Save it in a list
                self.bad_slots_truck_list.append( (self.slot, self.trucks[self.truck_ind], self.observation) ) 

                #if (distance_travelled > 427):
                    #self.reward += 10.0
                #if (distance_travelled_with_no_freight > 166):
                    #self.reward += 10.0
                #if (grams_of_CO2_emitted > 1_200_000):
                    #self.reward += 10.0

            elif (action == 1): #actions: 0 = bad choice, 1 = good choice, according to the agent
                #Save it in a list
                self.good_slots_truck_list.append( (self.slot, self.trucks[self.truck_ind], self.observation) ) 

                #if (distance_travelled < 427):
                    #self.reward += 10.0
                #if(distance_travelled_with_no_freight < 166):
                    #self.reward += 10.0
                #if (grams_of_CO2_emitted < 1_200_000):
                    #self.reward += 10.0
        
            
        ###########################Move to the next truck###########################
        
        self.truck_ind += 1 
        
        if (self.truck_ind >= number_of_trucks): #Every truck covered -> assign freight and move to the next one
            
            ###########################Assign this freight to the timeslot###########################
            
            if (self.good_slots_truck_list): #Check if there is at least one good slot 
                self.slots_truck_list = self.good_slots_truck_list
            elif (self.bad_slots_truck_list): #Else -> Check if there is at least one bad slot 
                self.slots_truck_list = self.bad_slots_truck_list
            
            #Check if there is at least one slot 
            if (self.slots_truck_list): #Slots available -> Let's pick randomly or manually
                self.best_slot_truck = random.choice(self.slots_truck_list)
                
                if (not self.auto): #Manual Matching
                    self.accepted_view = True
                    
                    self.window = tk.Tk()
                    self.window.geometry('1024x640')
                    self.window.title('Select your Preferred Matching')  
                    
                    self.label = tk.Label(self.window, text='Truck-Freight Matchings Accepted by the AI', fg = 'green', font=("Helvetica", 24))
                    self.label.pack()
                    
                    observations = []
                    self.current_trucks = []
                    self.current_slots = []
                    
                    for slot in self.good_slots_truck_list:
                        observations.append(slot[2])
                        self.current_trucks.append(slot[1])
                        self.current_slots.append(slot[0])
                        
                    #treeview
                    self.table = ttk.Treeview(self.window, columns = ('distance', 'distance_no_freight', 'CO2'), show = 'headings')
                    self.table.heading('distance', text = 'Distance Travelled (Km)')
                    self.table.heading('distance_no_freight', text = 'Distance Travelled with no Freight (Km)')
                    self.table.heading('CO2', text = 'Grams of CO2 Emitted')
                    self.table.pack(fill = 'both', expand = True)
                    
                    #Insert values
                    for obs in observations:
                        obs = np.round(np.exp(obs) - 1, 1) #Retrieve the original value
                        self.table.insert(parent = '', index = tk.END, values = (obs[0], obs[1], f"{obs[2]:_}"))
                    
                    #Table Events
                    self.table.bind('<<TreeviewSelect>>', self.item_select)
                    
                    self.b1 = tk.Button(self.window, text='Switch to Rejected Matchings', fg='red', font=("Helvetica", 16))
                    self.b1.bind("<Button-1>", self.on_left_click_switch)
                    self.b1.pack()

                    #self.b2 = tk.Button(self.window, text='Manual (click to switch to Auto)', font=("Helvetica", 16))
                    #self.b2.bind("<Button-1>", self.on_left_click_auto)
                    #self.b2.pack()
                    
                    self.b2 = tk.Button(self.window, text='Exit Program', font=("Helvetica", 16))
                    self.b2.bind("<Button-1>", self.on_left_click_exit)
                    self.b2.pack()

                    self.window.mainloop()
                    
                best_truck = self.best_slot_truck[1]
                best_slot = self.best_slot_truck[0]

                freight = self.freight_list[self.freight_ind] #Get the current freight
                freight.assignToTruck(best_truck.getTruckId())

                freight_id = self.freight_ind
                freight_destination = freight.getDestination()
                
                timeslot = best_slot["time_slot"]
                distance_travelled = best_slot["distance_travelled"]
                distance_travelled_with_no_freight = best_slot["distance_travelled_with_no_freight"]
                time_spent = best_slot["travel_time"]
                time_spent_with_no_freight = best_slot["time_spent_with_no_freight"]
                grams_of_CO2_emitted = best_slot["grams_of_CO2_emitted"]
                
                timeslot.freightAssignedToTruck(freight_id, distance_travelled, distance_travelled_with_no_freight, time_spent, time_spent_with_no_freight, grams_of_CO2_emitted)

                #Update the truck's location 
                best_truck.updateLocation(freight_destination)
        
            ###########################Move to the next freight###########################
            
            self.slots_truck_list = []
            self.bad_slots_truck_list = []
            self.good_slots_truck_list = []
        
            self.freight_ind += 1
            self.truck_ind = 0
        
            if (self.freight_ind >= number_of_freight): #Every freight covered -> move to the next day
                self.day += 1
                self.freight_ind = 0

                if (self.day >= number_of_days): #We've went through every day
                    self.done = True
                    self.observation = [-1.0, -1.0, -1.0]

                    calculateStatistics(self.trucks)
                
                    self.observation = np.array(self.observation)
                
                    return self.observation, self.reward, self.done, self.truncated, self.info
                
        truck = self.trucks[self.truck_ind] #Get the current truck
        freight = self.freight_list[self.freight_ind] #Get the current freight
        
        if ( not freight.isAssignedToTruck() ):
        
            freight_id = freight.getId()

            freight_category = freight.getCategory()
            freight_weight = freight.getWeight()
            freight_length = freight.getLength()
            freight_width = freight.getWidth()
            freight_height = freight.getHeight()

            freight_arrival = freight.getArrival()
            freight_deadline = freight.getDeadline()
            freight_origin = freight.getOrigin()
            freight_destination = freight.getDestination()
            
            freight_tags = freight.getTags()

            if (truck.containerCanHoldfreight(freight_category, freight_weight, freight_length, freight_width, freight_height, freight_tags)):
                self.slot = truck.timeSlotAvailableForDelivery(self.day, freight_arrival, freight_deadline, freight_origin, freight_destination, freight_weight)
                if (self.slot): #This slot is available
                    
                    #This is the observation
                    self.observation = [ (np.log(self.slot["distance_travelled"] + 1)), (np.log(self.slot["distance_travelled_with_no_freight"] + 1)), (np.log(self.slot["grams_of_CO2_emitted"] + 1)) ]
                    
                else: #This slot is not available
                    self.observation = [-1.0, -1.0, -1.0]
                    self.slot = {}
            else: #The container can't hold the freight
                self.observation = [-1.0, -1.0, -1.0]
                self.slot = {}
        else: #The freight is already assigned
            self.observation = [-1.0, -1.0, -1.0]
            self.slot = {}
                    
        self.observation = np.array(self.observation)
        
        return self.observation, self.reward, self.done, self.truncated, self.info

    def reset(self, seed=None, options=None):
        self.auto = False
        
        self.done = False
        self.reward = 0.0
        self.observation = [-1.0, -1.0, -1.0]
        
        self.trucks = generateRandomTrucks(number_of_trucks)
        self.freight_list = generateRandomFreight(number_of_freight)
        
        self.day = 0
        self.freight_ind = 0 #Get the first freight 
        self.truck_ind = 0 #Get The first truck
        
        self.slots_truck_list = []
        self.bad_slots_truck_list = []
        self.good_slots_truck_list = []
                
        ##############################################The first truck and the first freight

        truck = self.trucks[self.truck_ind] #Get the first Truck
        freight = self.freight_list[self.freight_ind] #Get the first freight
        
        freight_id = freight.getId()

        freight_category = freight.getCategory()
        freight_weight = freight.getWeight()
        freight_length = freight.getLength()
        freight_width = freight.getWidth()
        freight_height = freight.getHeight()

        freight_arrival = freight.getArrival()
        freight_deadline = freight.getDeadline()
        freight_origin = freight.getOrigin()
        freight_destination = freight.getDestination()
        
        freight_tags = freight.getTags()
        
        self.slot = {}

        if (truck.containerCanHoldfreight(freight_category, freight_weight, freight_length, freight_width, freight_height, freight_tags)):
            self.slot = truck.timeSlotAvailableForDelivery(self.day, freight_arrival, freight_deadline, freight_origin, freight_destination, freight_weight)
            if (self.slot): #This slot is available
                
                #This is the observation
                self.observation = [ (np.log(self.slot["distance_travelled"] + 1)), (np.log(self.slot["distance_travelled_with_no_freight"] + 1)), (np.log(self.slot["grams_of_CO2_emitted"] + 1)) ]
                
            else: #This slot is not available
                self.observation = [-1.0, -1.0, -1.0]
                self.slot = {}
        else: #The container can't hold the freight
            self.observation = [-1.0, -1.0, -1.0]
            self.slot = {}
                
        
        self.info = {}
        self.truncated = False
        self.observation = np.array(self.observation)
        
        return self.observation, self.info

## Add the environment to the gym registry

In [10]:
gym.register(
    id = "custom/FreightTruckMatching-v1",
    entry_point = FreightTruckMatchingEnv,
    max_episode_steps = number_of_trucks * number_of_freight * number_of_days,
)

## Make sure the environment is valid

In [11]:
#env = FreightTruckMatchingEnv()
env = gym.make("custom/FreightTruckMatching-v1")

# It will check your custom environment and output additional warnings if needed
check_env(env)

## Run it for a couple episodes

In [52]:
#Run it for a couple episodes with random actions to see if there are any errors

episodes = 20

for episode in range(episodes):
    print("\n\n############Episode:", episode, "############\n")
    
    done = False
    obs = env.reset()
    while not done:
        random_action = env.action_space.sample()
        #print("action:", random_action)
        obs, reward, done, truncated, info = env.step(random_action)
        #print('reward:', reward, "\n")



############Episode: 0 ############



KeyboardInterrupt: 

## AI Algorithm - PPO

In [17]:
models_dir = f"models/{int(time.time())}/"
logdir = f"logs/{int(time.time())}/"

if not os.path.exists(models_dir):
    os.makedirs(models_dir)

if not os.path.exists(logdir):
    os.makedirs(logdir)

env.reset()

model = PPO('MlpPolicy', env, verbose=0, tensorboard_log=logdir)

TIMESTEPS = 10000
iters = 0
while True:
    iters += 1
    model.learn(total_timesteps=TIMESTEPS, reset_num_timesteps=False, tb_log_name=f"AI ALgorithm 2")
    model.save(f"{models_dir}/{TIMESTEPS*iters}")

total_distance_travelled: 86946.3 Km
total_distance_travelled_with_no_freight: 36845.7 Km

total_time_spent: 36 days, 21:10:00
total_time_spent_with_no_freight: 15 days, 19:20:00

total_grams_of_CO2_emitted: 259_988_394 grams

number_of_freight_delivered: 207
percentage of freight delivered: 82.8 %



total_distance_travelled: 90417.0 Km
total_distance_travelled_with_no_freight: 39511.9 Km

total_time_spent: 37 days, 23:48:00
total_time_spent_with_no_freight: 16 days, 16:37:00

total_grams_of_CO2_emitted: 252_626_899 grams

number_of_freight_delivered: 199
percentage of freight delivered: 79.6 %



total_distance_travelled: 84624.6 Km
total_distance_travelled_with_no_freight: 30326.0 Km

total_time_spent: 35 days, 17:44:00
total_time_spent_with_no_freight: 13 days, 5:25:00

total_grams_of_CO2_emitted: 277_682_007 grams

number_of_freight_delivered: 203
percentage of freight delivered: 81.2 %



total_distance_travelled: 79830.6 Km
total_distance_travelled_with_no_freight: 27885.9 Km

t

KeyboardInterrupt: 

## Load the Model

In [14]:
models_dir = "models/1746023243/"
model_path = f"{models_dir}/1600000.zip"

model = PPO.load(model_path, env=env)

episodes = 3

for ep in range(episodes):
    obs, _ = env.reset()
    
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)

total_distance_travelled: 75092.8 Km
total_distance_travelled_with_no_freight: 23589.4 Km

total_time_spent: 31 days, 9:09:00
total_time_spent_with_no_freight: 10 days, 3:01:00

total_grams_of_CO2_emitted: 265_784_003 grams

number_of_freight_delivered: 198
percentage of freight delivered: 79.2 %



total_distance_travelled: 69338.3 Km
total_distance_travelled_with_no_freight: 20727.3 Km

total_time_spent: 29 days, 16:09:00
total_time_spent_with_no_freight: 9 days, 12:24:00

total_grams_of_CO2_emitted: 243_916_546 grams

number_of_freight_delivered: 188
percentage of freight delivered: 75.2 %



total_distance_travelled: 68055.1 Km
total_distance_travelled_with_no_freight: 21291.0 Km

total_time_spent: 29 days, 4:36:00
total_time_spent_with_no_freight: 9 days, 8:37:00

total_grams_of_CO2_emitted: 221_118_586 grams

number_of_freight_delivered: 197
percentage of freight delivered: 78.8 %





In [16]:
models_dir = "models/1746025626/"
model_path = f"{models_dir}/800000.zip"

model = PPO.load(model_path, env=env)

episodes = 3

for ep in range(episodes):
    obs, _ = env.reset()
    
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)

total_distance_travelled: 72228.7 Km
total_distance_travelled_with_no_freight: 20270.1 Km

total_time_spent: 30 days, 7:03:00
total_time_spent_with_no_freight: 8 days, 21:56:00

total_grams_of_CO2_emitted: 253_253_303 grams

number_of_freight_delivered: 191
percentage of freight delivered: 76.4 %



total_distance_travelled: 69426.0 Km
total_distance_travelled_with_no_freight: 21457.2 Km

total_time_spent: 29 days, 21:14:00
total_time_spent_with_no_freight: 9 days, 14:56:00

total_grams_of_CO2_emitted: 253_898_324 grams

number_of_freight_delivered: 197
percentage of freight delivered: 78.8 %



total_distance_travelled: 73217.4 Km
total_distance_travelled_with_no_freight: 20768.0 Km

total_time_spent: 30 days, 22:22:00
total_time_spent_with_no_freight: 9 days, 2:01:00

total_grams_of_CO2_emitted: 252_251_062 grams

number_of_freight_delivered: 200
percentage of freight delivered: 80.0 %





In [20]:
models_dir = "models/1746027912/"
model_path = f"{models_dir}/1400000.zip"

model = PPO.load(model_path, env=env)

episodes = 3

for ep in range(episodes):
    obs, _ = env.reset()
    
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)

total_distance_travelled: 83449.3 Km
total_distance_travelled_with_no_freight: 31738.6 Km

total_time_spent: 35 days, 10:36:00
total_time_spent_with_no_freight: 13 days, 18:41:00

total_grams_of_CO2_emitted: 212_353_540 grams

number_of_freight_delivered: 199
percentage of freight delivered: 79.6 %



total_distance_travelled: 81538.7 Km
total_distance_travelled_with_no_freight: 31389.5 Km

total_time_spent: 34 days, 23:41:00
total_time_spent_with_no_freight: 13 days, 23:46:00

total_grams_of_CO2_emitted: 251_756_374 grams

number_of_freight_delivered: 195
percentage of freight delivered: 78.0 %



total_distance_travelled: 86900.4 Km
total_distance_travelled_with_no_freight: 33132.4 Km

total_time_spent: 36 days, 21:42:00
total_time_spent_with_no_freight: 14 days, 14:22:00

total_grams_of_CO2_emitted: 263_236_283 grams

number_of_freight_delivered: 206
percentage of freight delivered: 82.4 %





# Explainability

## Load the Model

In [12]:
models_dir = "models/1746025626/"
model_path = f"{models_dir}/800000.zip"

model = PPO.load(model_path, env=env)

episodes = 1

for ep in range(episodes):
    obs, _ = env.reset()
    
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)